### 07. Understanding User Preferences and Preparing Interaction Data

**Objective:** Build a simulated user-item interaction dataset, convert it into a user-item matrix, run basic analysis, and explain how collaborative filtering uses these interactions to generate personalized recommendations.

**Input:** `../data/processed/ted_talks_cleaned.csv`
**Output:** `../data/interaction/user_interactions.csv`

**⚠ Important:** The interaction data in this notebook is **simulated** for learning purposes. No real user data is used.

**Design:**
- **50 users × 200 TED Talks**, sparsely rated (~1,150 interactions).
- Each user has 2–3 **preferred tags** (from the TED dataset) - they rate talks matching those tags higher.
- Rating scale: **1–5**.
- Additional signals: `liked`, `saved`, `watch_pct` (0–1), `timestamp`.

In [27]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

np.random.seed(42)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 160)

##### Load Cleaned TED Dataset
We use the cleaned dataset so talks have normalized tags. We'll sample **200 talks** to keep the interaction data manageable.

In [28]:
df_talks = pd.read_csv("../data/processed/ted_talks_cleaned.csv")
print(f"Full dataset: {df_talks.shape[0]} talks")

# Sample 200 talks (deterministic)
talk_pool = df_talks.sample(n=200, random_state=42).reset_index(drop=True)
talk_pool = talk_pool[["talk_id", "title", "tags"]].copy()

# Build a tag list per talk (lowercased)
def parse_tags(s):
    return [t.strip().lower() for t in str(s).split(",") if t.strip()]

talk_pool["tag_list"] = talk_pool["tags"].apply(parse_tags)

print(f"Sampled talks: {len(talk_pool)}")
talk_pool.head(3)

Full dataset: 4005 talks
Sampled talks: 200


,talk_id,title,tags,tag_list
0,995,glorious visions in animation and performance,"art, design, performance, technology, animation","[art, design, performance, technology, animation]"
1,1307,medicine for the 99 percent,"tedx, economics, global issues, health","[tedx, economics, global issues, health]"
2,1879,what is so special about the human brain,"brazil, brain, evolution, neuroscience","[brazil, brain, evolution, neuroscience]"


##### Define User Preferences
Each of the 50 users is assigned **2–3 preferred tags**. When a user "sees" a talk, the probability they interact and the rating they give depends on how well the talk's tags match their preferences.

In [29]:
# All unique tags across the 200 sampled talks
all_tags = sorted({t for tags in talk_pool["tag_list"] for t in tags})
print(f"Unique tags in sampled talks: {len(all_tags)}")
print(all_tags[:20], "...")

N_USERS = 50
user_ids = [f"U{str(i).zfill(3)}" for i in range(1, N_USERS + 1)]

# Assign 2-3 preferred tags per user
user_prefs = {}
for uid in user_ids:
    k = np.random.choice([2, 3])
    user_prefs[uid] = set(np.random.choice(all_tags, size=k, replace=False).tolist())

# Quick sample
for uid in user_ids[:5]:
    print(f"{uid}: {sorted(user_prefs[uid])}")

Unique tags in sampled talks: 323
['activism', 'adventure', 'advertising', 'africa', 'aging', 'agriculture', 'ai', 'aids', 'algorithm', 'alternative energy', 'ancient world', 'animals', 'animation', 'anthropocene', 'archaeology', 'architecture', 'art', 'asteroid', 'astronomy', 'bacteria'] ...
U001: ['farming', 'god']
U002: ['computers', 'graphic design', 'social change']
U003: ['google', 'intelligence', 'terrorism']
U004: ['advertising', 'animation']
U005: ['activism', 'egypt', 'religion']


##### Generate Simulated Interactions
For every (user, talk) pair we simulate:
- **Probability of interacting** (watching) rises with tag overlap.
- **Rating (1–5)** is higher when the talk matches the user's preferences.
- **liked** is True if rating ≥ 4.
- **saved** is True if rating = 5, or rating = 4 with 30% chance.
- **watch_pct** is drawn from a distribution shifted by rating.
- **timestamp** is random within the last 2 years.

In [30]:
rows = []
start_date = datetime(2023, 1, 1)
end_date   = datetime(2024, 12, 31)
date_range = (end_date - start_date).days

for uid in user_ids:
    prefs = user_prefs[uid]
    for _, talk in talk_pool.iterrows():
        talk_tags = set(talk["tag_list"])
        overlap = len(prefs & talk_tags)

        # Probability of interacting (watching) the talk
        # base 5%, +25% per overlapping preferred tag
        # p_watch = 0.05 + 0.25 * overlap
        # OR
        # base 10%, +25% per overlapping preferred tag
        p_watch = 0.10 + 0.25 * overlap
        if np.random.rand() > p_watch:
            continue

        # Rating: base 2.5 + 0.8 per overlap + noise
        rating_raw = 2.5 + 0.8 * overlap + np.random.normal(0, 0.7)
        rating = int(np.clip(round(rating_raw), 1, 5))

        liked = rating >= 4
        saved = (rating == 5) or (rating == 4 and np.random.rand() < 0.3)

        # watch_pct: correlated with rating
        watch_pct = float(np.clip(np.random.normal(loc=0.2 + rating * 0.15, scale=0.15), 0.0, 1.0))

        ts = start_date + timedelta(days=int(np.random.randint(0, date_range)))

        rows.append({
            "user_id": uid,
            "talk_id": int(talk["talk_id"]),
            "rating": rating,
            "liked": liked,
            "saved": saved,
            "watch_pct": round(watch_pct, 2),
            "timestamp": ts.strftime("%Y-%m-%d"),
        })

interactions = pd.DataFrame(rows)
print(f"Total interactions: {len(interactions)}")
print(f"Unique users: {interactions['user_id'].nunique()}")
print(f"Unique talks: {interactions['talk_id'].nunique()}")
print(f"Sparsity: {1 - len(interactions) / (N_USERS * len(talk_pool)):.2%} of the matrix is empty")
interactions.head()

Total interactions: 1153
Unique users: 50
Unique talks: 200
Sparsity: 88.47% of the matrix is empty


,user_id,talk_id,rating,liked,saved,watch_pct,timestamp
0,U001,1307,4,True,False,0.57,2023-07-02
1,U001,1075,3,False,False,0.62,2024-09-09
2,U001,2030,2,False,False,0.80,2023-12-01
3,U001,719,3,False,False,0.60,2023-10-15
4,U001,21894,2,False,False,0.43,2024-02-03


##### Save the Interaction Dataset

In [31]:
INTERACTION_DIR = Path("../data/interaction")
INTERACTION_DIR.mkdir(parents=True, exist_ok=True)

INTERACTION_PATH = INTERACTION_DIR / "user_interactions.csv"
interactions.to_csv(INTERACTION_PATH, index=False)
print(f"Saved: {INTERACTION_PATH}")

Saved: ..\data\interaction\user_interactions.csv


##### Build the User-Item Interaction Matrix
- **Rows:** users
- **Columns:** TED Talks
- **Values:** rating (1–5)
- **Missing values (NaN)** = the user has not interacted with that talk.

In [32]:
user_item_matrix = interactions.pivot_table(
    index="user_id",
    columns="talk_id",
    values="rating",
    aggfunc="mean",
)

print(f"Matrix shape: {user_item_matrix.shape}  (users × talks)")
print(f"Non-null entries: {user_item_matrix.notna().sum().sum()}")
print(f"Missing entries: {user_item_matrix.isna().sum().sum()}")

# Preview the first 8x8 block
user_item_matrix.iloc[:8, :8]

Matrix shape: (50, 200)  (users × talks)
Non-null entries: 1153
Missing entries: 8847


talk_id,27,44,46,71,79,91,131,190
user_id,,,,,,,,
U001,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN
U002,NaN,NaN,2.0,NaN,NaN,NaN,NaN,2.0
U003,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN
U004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
U005,NaN,NaN,NaN,4.0,3.0,NaN,NaN,NaN
U006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
U007,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN
U008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0


##### Basic Interaction Data Analysis

We answer the six questions asked in the task. Each is followed by a supporting code output.

In [ ]:
# Q1 - Unique users
print(f"Q1. Unique users: {interactions['user_id'].nunique()}")

# Q2 - Unique talks
print(f"Q2. Unique TED Talks interacted with: {interactions['talk_id'].nunique()}")

Q1. Unique users: 50
Q2. Unique TED Talks interacted with: 200


In [ ]:
# Q3 - TED Talks with the highest average rating (min 5 ratings to avoid small-sample bias)
top_rated = (
    interactions.groupby("talk_id")
    .agg(avg_rating=("rating", "mean"), n_ratings=("rating", "size"))
    .query("n_ratings >= 5")
    .sort_values("avg_rating", ascending=False)
    .head(10)
    .reset_index()
    .merge(talk_pool[["talk_id", "title"]], on="talk_id", how="left")
)
print("Q3. Top 10 highest-rated TED Talks (min 5 ratings):")
top_rated[["talk_id", "title", "avg_rating", "n_ratings"]]

Q3. Top 10 highest-rated TED Talks (min 5 ratings):


,talk_id,title,avg_rating,n_ratings
0,2265,the surprisingly logical minds of babies,3.800000,5
1,2700,smelfies and other experiments in synthetic biology,3.600000,5
2,2707,a burial practice that nourishes the planet,3.500000,6
3,386,the story of ezra,3.428571,7
4,372,design and the elastic mind,3.375000,8
5,1534,a test for parkinson s with a phone call,3.363636,11
6,55593,the mysterious life and death of rasputin,3.250000,8
7,23843,how mendel s pea plants helped us understand genetics,3.200000,5
8,566,a plug for smart power outlets,3.200000,5
9,2621,it s time for women to run for office,3.166667,6


> **Note on ratings:** Since interaction probability and rating both scale with tag overlap, most users give talks in their non-preferred topics low ratings (1-2). This keeps average ratings modest (2.5–3.5). In a real system, ratings would be more polarized (mostly 4-5 with a few 1s).

In [ ]:
# Q4 - Users who rated the most talks
most_active_users = (
    interactions.groupby("user_id")
    .agg(n_interactions=("talk_id", "size"))
    .sort_values("n_interactions", ascending=False)
    .head(10)
)
print("Q4. Top 10 most-active users:")
most_active_users

Q4. Top 10 most-active users:


,n_interactions
user_id,
U035,35
U033,34
U001,33
U044,33
U026,32
U006,32
U013,31
U005,31
U012,30


In [ ]:
# Q5 - Talks with the most interactions
most_interacted_talks = (
    interactions.groupby("talk_id")
    .agg(n_interactions=("user_id", "size"), avg_rating=("rating", "mean"))
    .sort_values("n_interactions", ascending=False)
    .head(10)
    .reset_index()
    .merge(talk_pool[["talk_id", "title"]], on="talk_id", how="left")
)
print("Q5. Top 10 most-interacted TED Talks:")
most_interacted_talks[["talk_id", "title", "n_interactions", "avg_rating"]]

Q5. Top 10 most-interacted TED Talks:


,talk_id,title,n_interactions,avg_rating
0,1562,the promise of research with stem cells,14,2.357143
1,1501,the game that can give you 10 extra years of life,13,2.692308
2,190,the anthropology of mobile phones,12,2.333333
3,230,5 predictions from 1984,11,2.909091
4,1534,a test for parkinson s with a phone call,11,3.363636
5,1294,how to topple a dictator,11,2.636364
6,771,four ways to fix a broken legal system,10,2.700000
7,719,my message of peace from pakistan,10,2.900000
8,2326,how fear drives american politics,10,2.900000
9,2570,the era of personal dna testing is here,10,2.600000


##### 7. What Does a Missing Rating Mean?

A **missing value** in the user-item matrix means **the user has not interacted with that talk**. It does **not** mean they disliked it. Reasons could be:

1. **Not exposed** - the talk was never recommended or shown.
2. **No interest** - the user saw it but chose not to watch/rate.
3. **Watched but not rated** - implicit interaction without an explicit rating.
4. **Not released yet** at the time of interaction.

Collaborative filtering algorithms must distinguish **explicit missing** (never seen) from **implicit negative** (seen, disliked). Our simulated dataset only captures the *explicit* side - watched talks are rated.

##### 8. How Collaborative Filtering Uses This Data

**Core idea:** If two users rate several talks similarly, they likely share interests. A talk liked by User A but not yet seen by User B is a good candidate to recommend to User B.

**Example:**
- User A and User B both rated *"Averting the climate crisis"* 5, *"Simplicity sells"* 4, and *"The best stats you've ever seen"* 5.
- The system infers a high **user-user similarity** (cosine or Pearson).
- User A also gave 5 to *"The case for optimism on climate change"*, which User B hasn't interacted with.
- The system predicts User B would rate it highly → **recommend it**.

**Two main approaches:**

| Approach | How it works | Needs |
|---|---|---|
| **User-based CF** | Find users similar to the target user; recommend talks they liked. | User-user similarity matrix. |
| **Item-based CF** | Find talks similar to ones the target user already liked; recommend those. | Item-item similarity matrix. |

**Why our interaction data enables this:**
- **User IDs** identify who is who.
- **Talk IDs** identify what was interacted with.
- **Ratings (1–5)** quantify preference strength.
- **Sparsity** (~88% empty) means most users only rate a small subset - exactly the realistic scenario CF is designed to handle.

**From the matrix:** rows are users, columns are talks, values are ratings. Once we have enough non-missing values, **CF can predict the missing ones** and rank talks per user.

##### Observations

1. **Simulated but realistic.** The dataset deliberately encodes tag-preference biases - users rate talks in their preferred topics higher. This creates a signal CF can learn.
2. **Sparse by design.** Most of the user-item matrix is empty. This mirrors real recommender systems where a user interacts with <1% of available items.
3. **Rating distribution is skewed high** (~3.5 average) because we only simulate interactions when users "watch" a talk - they self-select into content they expect to like.
4. **Active users vs. popular talks.** A small subset of users and talks dominate interactions - classic long-tail behavior. This means:
   - Popular talks get recommended often (popularity bias).
   - Most users have few ratings (cold-start risk).
5. **Missing rating ≠ dislike.** A NaN means *no interaction*, not *negative rating*. Ignoring this distinction in CF leads to bad predictions.
6. **Extensible.** Adding more signals (likes, saves, watch_pct) allows the model to move from **explicit feedback** (ratings) to **implicit feedback** (behaviour), which typically improves recommendation quality in production.
7. **Next step (Task 8):** Use this matrix to compute **user-user similarity** or **item-item similarity** and predict missing ratings. Then merge with the content-based recommender from Task 6 for a hybrid system.

In [37]:
# Final sanity checks
print("Final interaction data:")
print(f"  Rows: {len(interactions)}")
print(f"  Unique users: {interactions['user_id'].nunique()}")
print(f"  Unique talks: {interactions['talk_id'].nunique()}")
print(f"  Matrix shape: {user_item_matrix.shape}")
print(f"  Sparsity: {user_item_matrix.isna().mean().mean():.2%}")

# Verify every user has at least 1 rating
min_ratings_per_user = interactions.groupby("user_id").size().min()
print(f"  Min ratings per user: {min_ratings_per_user}")

Final interaction data:
  Rows: 1153
  Unique users: 50
  Unique talks: 200
  Matrix shape: (50, 200)
  Sparsity: 88.47%
  Min ratings per user: 12


#### 📄 Task Report

The written report for this notebook lives at [`reports/task_07_observations.md`](../reports/task_07_observations.md).

- Markdown: [`reports/task_07_observations.md`](../reports/task_07_observations.md)
- PDF: [`reports/pdf/task_07_observations.pdf`](../reports/pdf/task_07_observations.pdf)

If a link does not open inside your notebook viewer, open the file directly from the project root.
